In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [2]:
columns_to_use = ['V1AF02', 'V1AF14', 'V1AF10','PublicID'] # years of education, family income, English
df1 = pd.read_csv('../../Data/V1A.csv', usecols=columns_to_use,encoding='latin-1')

df2 = pd.read_csv('../../Data/DEMOGRAPHICS.CSV', usecols=['Age_at_V1','PublicID','CRace'],encoding='latin-1')
merged_df = pd.merge(df2, df1, on='PublicID', how='inner')

# Convert the selected demographic and survey columns to integers for modeling.
columns_to_convert = ['Age_at_V1', 'V1AF02', 'V1AF10']
merged_df[columns_to_convert] = merged_df[columns_to_convert].apply(
    pd.to_numeric, errors='coerce'
)

In [3]:
# Read the healthcare-response columns that will be collapsed into one flag.
healthcare_columns = ['V1AF15a', 'V1AF15b', 'V1AF15c', 'V1AF15d', 'V1AF15e', 'V1AF15f', 'V1AF15g','PublicID']

# Create a row-level indicator showing whether any response implies healthcare access.
df = pd.read_csv('../../Data/V1A.csv', usecols=healthcare_columns)

df['has_healthcare'] = 0

for index, row in df.iterrows():
    for column in healthcare_columns:
        if row[column] == 1:
            df.at[index, 'has_healthcare'] = 1  # 'Has_Healthcare' 1 if individual has healthcare
            break
        elif row[column] == 0:
            df.at[index, 'has_healthcare'] = 0  # 'Has_Healthcare' 0 if individual does not have healthcare
            break


In [4]:
merged_df = pd.merge(merged_df, df[['PublicID','has_healthcare']], on='PublicID', how='inner')

In [5]:
df_outcomes = pd.read_csv('../../Data/modified/Outcome.csv', usecols=['PublicID','MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

merged_df = pd.merge(merged_df, df_outcomes, on='PublicID', how='inner')
merged_df

,PublicID,Age_at_V1,CRace,V1AF02,V1AF10,V1AF14,has_healthcare,MH_outcome
0,00004O,21.0,3.0,4.0,1.0,5,0,1
1,00007I,19.0,1.0,4.0,1.0,4,1,1
2,00008G,30.0,1.0,3.0,1.0,10,0,0
3,00015J,32.0,1.0,7.0,1.0,12,0,0
4,00016H,21.0,2.0,3.0,1.0,7,1,1
...,...,...,...,...,...,...,...,...
7736,17349I,23.0,1.0,4.0,1.0,3,0,1
7737,17350A,22.0,1.0,4.0,1.0,4,1,1
7738,17351V,40.0,1.0,7.0,1.0,11,0,0
7739,17352T,31.0,4.0,8.0,1.0,13,0,1


In [6]:
# Remove non-response codes before modeling.
merged_df = merged_df[merged_df['V1AF14'] != 'D']
merged_df = merged_df[merged_df['V1AF14'] != 'R']

In [7]:

X = merged_df.drop(['MH_outcome', 'PublicID'], axis=1)
y = merged_df['MH_outcome']

train_df = merged_df[merged_df['PublicID'].isin(train_ids)].copy()
test_df = merged_df[merged_df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    3907
1    2504
Name: count, dtype: int64

In [8]:
numeric_features = ['Age_at_V1']
categorical_features = ['CRace']
binary_features = ['has_healthcare']
ordinal_features = ['V1AF02', 'V1AF14', 'V1AF10']

# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    binary_features=binary_features,
    ordinal_features=ordinal_features
)

Final dataset sizes for LR (impute=False): train=5135, test=1276
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 1, 'classifier__l1_ratio': 1.0}
Best CV Score (f1): 0.4895
Model Coefficients:
num__Age_at_V1: 0.12856941964247307
ord__V1AF02: -0.12963941686657363
ord__V1AF14: -0.07013966583047637
ord__V1AF10: -0.06452227979446418
bin__has_healthcare: 0.17032581525253743
cat__CRace_1.0: 0.0
cat__CRace_2.0: 0.005178151635814028
cat__CRace_3.0: -0.2343109361553167
cat__CRace_4.0: 0.26463956876361355
cat__CRace_5.0: -0.10625386922399832
Evaluation Metrics for LR with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5940
Precision (positive class): 0.4752
Recall (positive class): 0.4656
F1 (positive class): 0.4703
Macro Precision: 0.5709
Macro Recall: 0.5704
Macro F1: 0.5706
ROC AUC: 0.5927


In [9]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    binary_features=binary_features,
    ordinal_features=ordinal_features
)

Final dataset sizes for RF (impute=False): train=5135, test=1276
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 15, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 600}
Best CV Score (f1): 0.4488
Feature Importances:
num__Age_at_V1: 0.3828218564573697
ord__V1AF02: 0.1667543106305342
ord__V1AF14: 0.24758521499013525
ord__V1AF10: 0.03240077028383287
bin__has_healthcare: 0.06651534858945715
cat__CRace_1.0: 0.030462637494449364
cat__CRace_2.0: 0.019282114414744977
cat__CRace_3.0: 0.025458264980990682
cat__CRace_4.0: 0.011789834474592548
cat__CRace_5.0: 0.016929647683893222
Evaluation Metrics for RF with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5768
Precision (positive class): 0.4496
Recall (positive class): 0.4150
F1 (positive class): 0.4316
Macro Precision: 0.5486
Macro Recall: 0.5470
Macro F1: 0.5473
ROC AUC: 0.5701


In [10]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    binary_features=binary_features,
    ordinal_features=ordinal_features
)

Final dataset sizes for XGB (impute=False): train=5135, test=1276
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.001, 'classifier__max_depth': 4, 'classifier__n_estimators': 100, 'classifier__subsample': 0.5}
Best CV Score (f1): 0.4949
Feature Importances:
num__Age_at_V1: 0.07734806835651398
ord__V1AF02: 0.22725604474544525
ord__V1AF14: 0.1540636569261551
ord__V1AF10: 0.06964902579784393
bin__has_healthcare: 0.1932860016822815
cat__CRace_1.0: 0.04466366395354271
cat__CRace_2.0: 0.05373343825340271
cat__CRace_3.0: 0.07668741792440414
cat__CRace_4.0: 0.05604557320475578
cat__CRace_5.0: 0.047267138957977295
Evaluation Metrics for XGB with shared preprocessing and adaptive CV scoring:
Accuracy: 0.6050
Precision (positive class): 0.4902
Recall (positive class): 0.5040
F1 (positive class): 0.4970
Macro Precision: 0.5856
Macro Recall: 0.5864
Macro F1: 0.5859
ROC AUC: 0.6025


In [11]:
# Run the SVM pipeline with macro F1 grid search. (If output hidden, inspect the file itself with the "less" command)
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    binary_features=binary_features,
    ordinal_features=ordinal_features
)

Final dataset sizes for SVM (impute=False): train=5135, test=1276
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 'auto', 'classifier__estimator__kernel': 'rbf'}
Best CV Score (f1): 0.4913
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and adaptive CV scoring:
Accuracy: 0.6050
Precision (positive class): 0.4905
Recall (positive class): 0.5243
F1 (positive class): 0.5068
Macro Precision: 0.5882
Macro Recall: 0.5902
Macro F1: 0.5887
ROC AUC: 0.5984
